In [1]:
import yfinance as yf
import pandas as pd

df = yf.download("SPY", start="2015-01-01", end="2025-12-31", auto_adjust=True)

# normalise columns to what HistoricBarHandler expects
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)
df.columns = [c.lower() for c in df.columns]
df = df[["open", "high", "low", "close", "volume"]].dropna()
df.index.name = "date"

print(df.shape)
print(df.head())
df.to_parquet("../data/SPY_daily.parquet")   # or .to_csv if pyarrow failed

[*********************100%***********************]  1 of 1 completed

(2765, 5)
                  open        high         low       close     volume
date                                                                 
2015-01-02  170.472573  170.885580  168.655335  169.687851  121465900
2015-01-05  168.647112  168.812312  166.317762  166.623383  169632600
2015-01-06  166.928950  167.449342  164.260931  165.053909  209151400
2015-01-07  166.375567  167.449386  165.929525  167.110718  125346700
2015-01-08  168.514901  170.290837  168.498390  170.076065  147217800


In [ ]:
import sys; sys.path.insert(0, "../src")
import queue
from backtester.data import HistoricBarHandler

dh = HistoricBarHandler(df, "SPY")
ev = queue.Queue()
for _ in range(5):
    dh.update_bars(ev)
    print(ev.get().timestamp, dh.current_price())

2015-01-02 00:00:00 169.68785095214844
2015-01-05 00:00:00 166.62338256835938
2015-01-06 00:00:00 165.0539093017578
2015-01-07 00:00:00 167.1107177734375
2015-01-08 00:00:00 170.07606506347656


: 

In [ ]:
import pandas as pd
from backtester import metrics as m

df = pd.read_parquet("../data/SPY_daily.parquet")   # or read_csv

# Should be zero. If it isn't, fix data.py — don't work around it here.
print("NaN closes:", int(df["close"].isna().sum()), "of", len(df))

stats = m.summary(df["close"])

for k, v in stats.items():
    if k == "n_periods":
        print(f"{k:>16}: {v}")
    elif "return" in k or "drawdown" in k or k == "cagr":
        print(f"{k:>16}: {v:>8.2%}")
    else:
        print(f"{k:>16}: {v:>8.2f}")